In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads the .env file into environment variables

API_KEY = os.environ["GROQ_API_KEY"]

from openai import OpenAI
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


In [4]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'setup working' and nothing else."}]
)
print(resp.choices[0].message.content)

setup working


In [5]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content
answer = ask_llm("What is the capital of Ghana?")
print(answer)
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of Ghana?"},
    ],
)
print(response.usage)

The capital of Ghana is Accra.
CompletionUsage(completion_tokens=9, prompt_tokens=48, total_tokens=57, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.04107944, prompt_time=0.009355672, completion_time=0.00836087, total_time=0.017716542)


1. System is the set of instructions that define how a model should behave before the conversation begins while user is the actual task you ask a model in the moment.
Examples:
System: You are a loan assistant, hence use only facts from data supplied to you.
User: Summarize the document pasted below, include only important and urgent information in the summary.
2. Token is basically a small piece of text from a corpus that a model processes.
3. API provider charge per token because request can be widely different in size,computational and analytical cost. Charging per token requires a payment on how much text you actually used, thereby creating fairness.


In [6]:
question = "Suggest a name for a savings product for market traders in Accra."

print("Temperature 0.0")
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    print(f"{i+1}. {answer}")

print("\n=== Temperature 1.2 ===")
for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    print(f"{i+1}. {answer}")

Temperature 0.0
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders who need to manage their finances on-the-go.
5. **Sika Su**: "Sika" is the Ghanaian word for "money", and "Su" means "grow" or "increase", so this name suggests a savings product that helps traders grow their wealth.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word for "honest" or "trustworthy", which could convey a sense of reliability and security for market traders.
7. **Traders' Trust**: This name emphasizes the idea of tru

1. At temperature 0.0, there were some repetition(Makola, Accra Market, Savings,etc) in the answers across all 5 runs wjile at Temperature 1.2, the answers were quite varied, new words and ideas were introduced at each run. Therefore, for the loan decision support system, low temperature would be more suitable because the system needs to produce consistent and factual summaries that are very close to the application givint to the model. However, temperature alone doesnt guarantee factual accuracy nor prevent hallucination, hence the need for human oversight.

In [7]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [8]:
SUMMARY_PROMPT_V1 = "Summarize this: {letter_text}"

print("=== V1 on L002 ===")
print(ask_llm(SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L002"])))

print("\n=== V1 on L006 ===")
print(ask_llm(SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L006"])))

=== V1 on L002 ===
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but is optimistic it will improve after the festive season. He has no collateral to offer but promises to repay the loan as soon as possible.

=== V1 on L006 ===
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. Although he has no experience and no collateral, he claims to be "business-minded" and promises to repay the loan within a year, relying on his trustworthiness as assurance.


In [9]:
SUMMARY_SYSTEM_PROMPT_V2 = (
    "You are an assistant to a microfinance loan officer. "
    "Summarize loan applications factually and neutrally, in 3-4 sentences. "
    "Only include information explicitly stated in the letter. "
    "Do not invent, assume, or infer any details not present in the text. "
    "Do not offer opinions on whether the loan should be approved."
)

def summary_prompt_v2(letter_text):
    return f"Summarize this loan application:\n\n{letter_text}"

print("=== V2 on L002 ===")
print(ask_llm(
    summary_prompt_v2(LETTERS["L002"]),
    system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
    temperature=0
))

print("\n=== V2 on L006 ===")
print(ask_llm(
    summary_prompt_v2(LETTERS["L006"]),
    system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
    temperature=0
))

=== V2 on L002 ===
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but expects it to improve after the festive season. He does not currently have collateral to offer, but is requesting assistance with the loan.

=== V2 on L006 ===
Kofi is applying for a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He is 22 years old and claims to be "full of energy" and "business minded" as stated by his friends. Kofi has not yet started any of these businesses. He intends to repay the loan in one year and does not have collateral, but describes himself as trustworthy.


V1 somtimes changed the meaning of what the applicant actually said in their application letter. For example, with L002, it made Kwame's repayment statement to sound more reliable than it actually was. Meanwhile, there was uncertainty in his repayment statement. This problem is known as Hallucination.
V2 improved this by sticking to exactly what is exactly stated in the letter. Wording might change but it bulges down to the same meaning the applicant was conveying.
The instruction "no invented details" is crucial because the loan officer will rely on it to make final decision about money. For instance, changing an applicants true words to a differnt one, like what happened in L002's summary. The loan officer will approve the loan for Kwame with the intention that he will back as soon as possible.

In [10]:
EXTRACT_SYSTEM_PROMPT = """You are a data extraction assistant for a microfinance institution.
Extract the following fields from a loan application letter and return ONLY a JSON object
with exactly these keys, nothing else — no explanation, no markdown, no extra text:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

If a field is not explicitly stated in the letter, use null. Do not guess or infer.

Example:
Letter: "Hello, my name is Ama Serwaa, I sell fabrics at Kejetia Market. I need GHS 5,000
to buy new stock. I make about GHS 600 profit monthly. I have no guarantor yet. I can pay
back over 10 months."

Output:
{"applicant_name": "Ama Serwaa", "amount_ghs": 5000, "purpose": "buy new fabric stock",
"monthly_profit_ghs": 600, "has_collateral_or_guarantor": false, "repayment_months": 10}
"""

def extract_prompt(letter_text):
    return f"Letter:\n{letter_text}\n\nOutput:"

In [11]:
import json

def extract_fields(letter_text):
    raw_output = ask_llm(
        extract_prompt(letter_text),
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=0,
        max_tokens=300
    )
    
    
    cleaned = raw_output.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json", "", 1).strip()
    
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"Failed to parse JSON. Raw output was:\n{raw_output}")
        return None

In [12]:
import pandas as pd

results = {}
for letter_id, letter_text in LETTERS.items():
    print(f"Extracting {letter_id}...")
    results[letter_id] = extract_fields(letter_text)

df = pd.DataFrame.from_dict(results, orient="index")
df

Extracting L001...
Extracting L002...
Extracting L003...
Extracting L004...
Extracting L005...
Extracting L006...


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,buy feed and 500 new layers for poultry farm,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


1. The example should not come from the six letters because those letters are the data we are testing the model on. If we use one of them as the example, the model may memorize or copy information from the example instead of genuinely learning how to extract the fields.
2. "Use null, do not guess" is important because the model may try to fill in missing data based on assumptions. For example, L006 does not state Kofi's current monthly profit because he has not even started the businesses yet. The model should therefore return null rather than cooking a profit amount there. Without this instruction, an LLM may produce plausible-looking information that is not actually stated in the letter by the applicant.
3. Temperature 0 is suitable for extraction because we want the model to give consistent and fact from the main appliaction letter. The goal is not to be creative; it is to correctly identify information that already exists in the letter. A higher temperature introduces more variation, which is useful for only creative tasks.

In [15]:
BRIEF_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer in Ghana.
You will be given a loan application letter and structured data extracted from it.

Produce a decision-support brief with exactly these four sections:

1. Strengths — bullet points, grounded only in facts stated in the letter or data
2. Risks / Red Flags — bullet points, grounded only in facts stated in the letter or data
3. Missing Information — what the officer should request before deciding
4. Suggested Next Step — a PROCESS step only, such as "invite for interview",
   "request supporting documents", or "flag for senior review"

You must NOT output "approve", "reject", or any final lending decision. The final
decision must always be made by a human loan officer. Your job is only to support
that decision with an organized, factual brief.
"""

def brief_prompt(letter_text, extracted_json):
    return (
        f"Letter:\n{letter_text}\n\n"
        f"Extracted data:\n{json.dumps(extracted_json)}\n\n"
        f"Produce the decision-support brief."
    )
briefs = {}
for letter_id, letter_text in LETTERS.items():
    extracted = results[letter_id]
    brief = ask_llm(
        brief_prompt(letter_text, extracted),
        system_prompt=BRIEF_SYSTEM_PROMPT,
        temperature=0,
        max_tokens=500
    )
    briefs[letter_id] = brief

print("All briefs generated.")

for letter_id in ["L001", "L002","L003", "L006"]:
    print(f"=== Brief for {letter_id} ===")
    print(briefs[letter_id])
    print()

All briefs generated.
=== Brief for L001 ===
## Step 1: Strengths
The applicant has the following strengths:
* 12 years of experience selling provisions at Makola Market
* A stable monthly profit of GHS 900
* A history of saving with the susu scheme, having saved GHS 2,500 over two years without missing a contribution
* A guarantor, the applicant's sister, who is a teacher
* A clear plan for repayment, with a proposed monthly repayment amount of GHS 450 over 20 months

## Step 2: Risks / Red Flags
The applicant presents the following risks or red flags:
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit and savings
* The expansion into frozen foods may introduce new business risks, such as increased costs for maintenance and potential spoilage
* The repayment plan, although clear, may be challenging if the business does not generate sufficient profits

## Step 3: Missing Information
Before making a decision, the loan officer should request the foll

1. Yes, the system correctly identified the strengths and risks in both applications. For L003, it recognized strong evidence such as a registered business, good profits, collateral, and sales records. For L006, it identified major concerns such as no business experience, no collateral, and an uncertain repayment plan. It also did not make up strengths for L006. This shows that the system was able to tell the difference between claims that are supported by evidence and claims that are just someone's opinion or promise.

2. We stopped the model from saying "approve" or "reject" for two main reasons. Practically, AI can make mistakes, misunderstand information, or be biased. Since loans involve real money, a human loan officer should check the information and make the final decision. Ethically, loan decisions can greatly affect people's lives and opportunities. Keeping a human in charge helps ensure that decisions are fair, accountable, and not based only on an AI's judgment.

Commit hash: 
5459d9f38b492dded189a682f7f191a4bd88afd6 

In [16]:
fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
          "has_collateral_or_guarantor", "repayment_months"]

gold_letters = ["L001", "L003", "L006"]

comparison = {}

for field in fields:
    row = {}
    for letter_id in gold_letters:
        gold_value = GOLD[letter_id][field]
        extracted_value = results[letter_id][field]
        
        # Case-insensitive match for names, exact match for everything else
        if field == "applicant_name":
            match = str(gold_value).strip().lower() == str(extracted_value).strip().lower()
        else:
            match = gold_value == extracted_value
        
        row[letter_id] = "✅" if match else f"❌ (got: {extracted_value})"
    comparison[field] = row

comparison_df = pd.DataFrame.from_dict(comparison, orient="index")
comparison_df
accuracy = {}
for field in fields:
    correct_count = sum(1 for letter_id in gold_letters 
                         if comparison[field][letter_id] == "✅")
    accuracy[field] = f"{correct_count}/3 ({correct_count/3*100:.0f}%)"

comparison_df["accuracy"] = pd.Series(accuracy)
comparison_df

,L001,L003,L006,accuracy
applicant_name,✅,✅,✅,3/3 (100%)
amount_ghs,✅,✅,✅,3/3 (100%)
purpose,❌ (got: buy a deep freezer and expand into fro...,❌ (got: purchase two industrial sewing machine...,"❌ (got: start a car washing business, a provis...",0/3 (0%)
monthly_profit_ghs,✅,✅,✅,3/3 (100%)
has_collateral_or_guarantor,✅,✅,✅,3/3 (100%)
repayment_months,✅,✅,✅,3/3 (100%)


In [18]:
import json

def extract_fields(letter_text, temperature=0):
    raw_output = ask_llm(
        extract_prompt(letter_text),
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=temperature,
        max_tokens=300
    )
    
    cleaned = raw_output.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json", "", 1).strip()
    
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f" Failed to parse JSON. Raw output was:\n{raw_output}")
        return None

def run_reliability_test(letter_id, temperature, n_runs=5):
    outputs = []
    for i in range(n_runs):
        result = extract_fields(LETTERS[letter_id], temperature=temperature)
        outputs.append(result)
    return outputs

print("Running at temperature=0...")
runs_temp0 = run_reliability_test("L004", temperature=0)

print("Running at temperature=1.0...")
runs_temp1 = run_reliability_test("L004", temperature=1.0)

print("Done.")

Running at temperature=0...
Running at temperature=1.0...
Done.


In [19]:
def analyze_reliability(runs, label):
    valid_count = sum(1 for r in runs if r is not None)
    
    json_strings = [json.dumps(r, sort_keys=True) for r in runs if r is not None]
    unique_count = len(set(json_strings))
    
    print(f"=== {label} ===")
    print(f"Valid JSON: {valid_count}/5")
    print(f"Unique outputs: {unique_count} (1 = perfectly consistent, 5 = all different)")
    print()
    for i, r in enumerate(runs):
        print(f"Run {i+1}: {r}")
    print()

analyze_reliability(runs_temp0, "Temperature = 0")
analyze_reliability(runs_temp1, "Temperature = 1.0")

=== Temperature = 0 ===
Valid JSON: 5/5
Unique outputs: 1 (1 = perfectly consistent, 5 = all different)

Run 1: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'buy feed and 500 new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 2: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'buy feed and 500 new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 3: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'buy feed and 500 new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 4: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'buy feed and 500 new layers for poultry farm', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
Run 5: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'pu

In [20]:
test1_question = "What is Akosua Mensah's credit score, based on her loan application letter?"

test1_output = ask_llm(
    f"Here is a loan application:\n\n{LETTERS['L001']}\n\nQuestion: {test1_question}",
    system_prompt=SUMMARY_SYSTEM_PROMPT_V2,  # reuse your V2 summarizer's constraints
    temperature=0
)

print("=== Test 1: Asking about a detail NOT in the letter ===")
print(test1_output)

=== Test 1: Asking about a detail NOT in the letter ===
Akosua Mensah's loan application letter does not mention her credit score. The letter provides information about her business, savings, and proposed loan repayment plan, but it does not include any details about her credit score.


In [21]:
fake_weather_report = """
Today's weather in Accra: partly cloudy with a high of 31°C and a low of 24°C.
Expect scattered showers in the afternoon, clearing up by evening. Humidity
levels around 78%. Light winds from the southwest at 10 km/h.
"""

test2_output = extract_fields(fake_weather_report, temperature=0)

print("=== Test 2: Extracting from an irrelevant weather report ===")
print(test2_output)

=== Test 2: Extracting from an irrelevant weather report ===
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


1. The purpose field was the hardest to extract accurately. The other five fields were 100% accurate, but the purpose field received 0% when using exact matching. However, this does not mean the model extracted it incorrectly. The model gave answers with the same meaning but different wording from the gold-standard answer. This happened because purpose is a free-text field, so there can be many correct ways to describe the same thing. Exact matching is therefore not the best way to evaluate it; semantic similarity or human judgment would be more suitable.

2. At temperature 0, the model gave exactly the same results across all five runs. At temperature 1.0, there was some small variation, mainly in how the purpose was worded, but the numbers and boolean values stayed the same. This shows that temperature mainly affected the wording rather than the actual information extracted. However, temperature 0 is still safer for a production financial system because we want the same information to produce the same stored results every time.

3. No, the system did not hallucinate in either test. When asked for a credit score that was not included in the letter, the summarizer correctly said that the information was not provided instead of making up a score. When given an unrelated weather report, the extractor returned null for the missing loan information instead of inventing an applicant. This worked because the prompts clearly told the model not to guess or invent information and to use null when information was missing. For even more protection, the system could also use additional examples and an external validation step to check the model's output.

1. People who may be unfairly affected are applicants who have good businesses but struggle to write well in English. The system mainly uses the information clearly stated in the application, so important strengths might be missed if the applicant cannot explain them properly. This could make a person who writes well appear more worthy than someone with a poorly written application, even if the second person has a better business. Therefore, writing ability could become a factor in loan decisions, which would be unfair.

2. Loan applications contain private information such as names, income, savings, and business details. Sending this information to an API in another country means that the data is being shared with an outside company, which creates privacy concerns. Before using such a system in a real Ghanaian microfinance institution, I would check the provider's data retention and deletion policies, whether they use the data to train their models, how they protect the data, and whether applicants are informed and give consent. I would also check whether the data can be kept within Ghana or under appropriate legal protections.

3.  a. Mandatory human review: A human loan officer should always review the original application and the AI's output before making a final decision. The AI should only support the officer, not replace them.

     b. Audit logs and an appeal process: The system should keep records of the AI's outputs, the prompts used, and when the outputs were generated. Applicants should also be able to request a human review if they believe the AI misunderstood or misrepresented their application. This would help identify and correct mistakes or unfair treatment.